# Using Plotly

First, fetch the data. We use `pd.merge` to keep all datetimes, not only the common ones. This is more reasonable for plotting, as we can now see where we're missing what data.

In [105]:
import pandas as pd 
from get_table import get_table

# get hourly weather data for Heidelberg (2024-11-01 to 2025-10-31)
tue_weather_hourly = pd.read_csv('weather_data/hourly/tuebingen_weather_2024-11-01_2025-10-31.csv')

# get hourly bike count data for Heidelberg (2024-11-01 to 2025-10-31)
tue_bike_hourly = get_table('eco-counter/all_cities', 2024, 11, 1, 2025, 10, 31)
tue_bike_hourly = tue_bike_hourly[tue_bike_hourly['counter_site'] == "Fuß- & Radtunnel Südportal - Derendinger Allee"]

# Add a new column datetime in both tables (for same name)
tue_weather_hourly['datetime'] = pd.to_datetime(tue_weather_hourly['time'])
tue_bike_hourly['datetime'] = pd.to_datetime(tue_bike_hourly['iso_timestamp'])

# Merge both tables on datetime to get all data in one table
tue_df_common = pd.merge(
    tue_bike_hourly[['datetime', 'channels_all']],
    tue_weather_hourly[['datetime', 'prcp', 'temp']],
    on='datetime',
    how='outer'
).sort_values('datetime')

# Rename columns for clarity
tue_df_common.rename(columns={'channels_all': 'bike', 'prcp': 'rain', 'temp': 'temp'}, inplace=True)

tue_df_common[['bike', 'rain', 'temp']] = tue_df_common[['bike', 'rain', 'temp']].apply(
    pd.to_numeric, errors='coerce'
)


## Interactive plot of the data

- double click to zoom out completely 
- use rangeslider to zoom in
- hover vertically to see bike count, temperature, and rain of a specific datetime
- click on variable to show/hide the line in the plot


In [107]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Create interactive line plot with Plotly
fig = px.line(tue_df_common, x="datetime", y=tue_df_common.columns,
              category_orders={"variable": ["bike", "rain", "temp"]},
              color_discrete_map={    # Custom colors
                "bike": px.colors.qualitative.Set3[0],
                "temp": px.colors.qualitative.Set3[3],
                "rain": px.colors.qualitative.Set3[4],
                })

fig.update_layout(hovermode="x unified")        # On hover, show all values depending on x axis together
fig.update_traces(hovertemplate=None)           # Compact hover info

fig.update_xaxes(rangeslider_visible=True)      # Add range slider

fig.show()